In [ ]:
import SimpleITK as sitk
import numpy as np
import os

In [ ]:
def bounding_box(bladder_img, colon_img):

    bladder_array = sitk.GetArrayFromImage(bladder_img)  # Shape: [Z, Y, X]
    colon_array = sitk.GetArrayFromImage(colon_img)

    # Find the bladder mask Zmax
    bladder_indices = np.nonzero(bladder_array)
    Z_coords = bladder_indices[0]  # Since the array shape is [Z, Y, X]
    Zmax = np.max(Z_coords)

    # Create a mask for the rectum area
    rectum_array = colon_array.copy()
    rectum_array[Zmax:, :, :] = 0  # Assuming [Z, Y, X] ordering

    rectum_indices = np.nonzero(rectum_array)

    # Get the minimum and maximum indices for each dimension
    min_Z, max_Z = np.min(rectum_indices[0]), np.max(rectum_indices[0])
    min_Y, max_Y = np.min(rectum_indices[1]), np.max(rectum_indices[1])
    min_X, max_X = np.min(rectum_indices[2]), np.max(rectum_indices[2])

    # Define the expansion size (in voxels)
    expansion = 10

    # Expand the bounding box, ensuring it stays within image bounds
    min_Z = max(min_Z - 8*expansion, 0)
    max_Z = max_Z + expansion

    min_Y = max(min_Y - expansion, 0)
    max_Y = max_Y + expansion

    return min_Z, max_Z, min_Y, max_Y, min_X, max_X

In [ ]:
def extract_patches(minZ, maxZ, minY, maxY, minX, maxX, mr_image):
    # Define the size and index for the Region of Interest (ROI)
    roi_size = [
        int(maxX - minX + 1),
        int(maxY - minY + 1),
        int(maxZ - minZ + 1)
    ]

    roi_index = [
        int(minX),
        int(minY),
        int(minZ)
    ]

    # Set up the RegionOfInterestImageFilter
    roi_filter = sitk.RegionOfInterestImageFilter()
    roi_filter.SetSize(roi_size)
    roi_filter.SetIndex(roi_index)

    # Extract the patch
    rectum_patch = roi_filter.Execute(mr_image)
    return rectum_patch

In [ ]:
# Load the original MR image
path = r'/path/to/workspace/classificationmodel_original_sagittal/total_work/resampled'
save_path = r'/path/to/workspace/classificationmodel_original_sagittal/total_work/rectumcentrecrop_correct'

for i in os.listdir(path):
    foldername = i # patient name
    file_path = os.path.join(path, i)
    image_path =  os.path.join(file_path, 'resampled_image.nii.gz')
    bladder_path =  os.path.join(file_path, 'bladder.nii.gz')
    colon_path =  os.path.join(file_path, 'colon.nii.gz')

    image = sitk.ReadImage(image_path)
    bladder_img = sitk.ReadImage(bladder_path)
    colon_img = sitk.ReadImage(colon_path)

    try:
        min_Z, max_Z, min_Y, max_Y, min_X, max_X = bounding_box(bladder_img, colon_img)
        rectum_patch = extract_patches(min_Z, max_Z, min_Y, max_Y, min_X, max_X, image)
    
        # Save the extracted patch
        new_foldername = foldername + '.nii.gz'
        save_path_new = os.path.join(save_path, new_foldername)
        sitk.WriteImage(rectum_patch, save_path_new)
        
    except Exception as e:
        print(f"An error occurred: {e}")
        print("patient number:", foldername)
        
    
print('done')